# Analysis of cups geometrical features

## 1. Documentation and Imports


Created on 24-02-2021

Author: Valentin Laplaud

Measuring and summarizing cups geometrical features (decorations shape, cups shape)

In [ ]:

# plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt

# numbers handling
import numpy as np
import pandas as pd

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf

## 2. Define analysis functions

### 2.1 Compute decoration angle and height

Input : the dataframe from IJ analysis where there is two lines per decoration (angle and length of each side). This txt file is computed from the tif stack 'FullCrowns'

In [ ]:
def ComputeDeco(Data):
    L = len(Data)
    
    DecoCharac = pd.DataFrame(data=None) # initialize output dataframe
    
    # initialize variables
    DecoAngle = np.zeros(int(L/2))
    DecoHeight = np.zeros(int(L/2))
    StackSlice = np.zeros(int(L/2))
    
    # compute characteristics
    for i in range(0,L,2):
        
        DecoAngle[int(i/2)] = (Data.at[i, 'Angle'] - Data.at[i+1, 'Angle'])%180
        
        DecoHeight[int(i/2)] = np.cos(np.radians(DecoAngle[int(i/2)]/2))*(Data.at[i, 'Length'] + Data.at[i+1, 'Length'])/2
        
        StackSlice[int(i/2)] = Data.at[i, 'Slice']
        
    DecoCharac['Angle (deg)'] = DecoAngle
    DecoCharac['Height (µm)'] = DecoHeight
    DecoCharac['StackSlice'] = StackSlice.astype(int)
    
    return(DecoCharac)

### 2.2 Compute decoration finger length and width (of the first cell) 

Input the dataframe from IJ analysis where there is two lines per finger (length then width). This txt file is computed from the tif stack 'SuperZooms'

In [ ]:
def ComputeFinger(Data):
    L = len(Data)
    
    FingerCharac = pd.DataFrame(data=None) # initialize output dataframe
    
    # initialize variables
    FingerLength = np.zeros(int(L/2))
    FingerWidth = np.zeros(int(L/2))
    StackSlice = np.zeros(int(L/2))
    
    # compute characteristics
    for i in range(0,L,2):
        
        FingerLength[int(i/2)] = Data.at[i, 'Length']
        
        FingerWidth[int(i/2)] = Data.at[i+1, 'Length']
        
        StackSlice[int(i/2)] = Data.at[i, 'Slice']
        
    FingerCharac['Length (µm)'] = FingerLength
    FingerCharac['Width (µm)'] = FingerWidth
    FingerCharac['StackSlice'] = StackSlice.astype(int)
    
    return(FingerCharac)

# 3. Enter data and run

In [ ]:
# path to data
P = r'd:\Users\Valentin Laplaud\PostDoc\Data\Microscopy\2021.02.19\CupCaract\Crowns'

DecorationDataIJ = vf.LoadImageJResults(P + '\\DecorationAngles.txt',['Slice','Angle','Length'])
FingerDataIJ = vf.LoadImageJResults(P + '\\Fingers.txt',['Slice','Length'])

DecoCharac = ComputeDeco(DecorationDataIJ)
FingerCharac = ComputeFinger(FingerDataIJ)

# Write results

print('Decoration angle : ' + str(np.round(np.mean(DecoCharac['Angle (deg)'])*10)/10) +
      ' \u00B1 ' + str(np.round(np.std(DecoCharac['Angle (deg)']/np.sqrt(len(DecoCharac)))*10)/10) +  ' deg\n')

print('Decoration height : ' + str(np.round(np.mean(DecoCharac['Height (µm)'])*10)/10) +
      ' \u00B1 ' + str(np.round(np.std(DecoCharac['Height (µm)']/np.sqrt(len(DecoCharac)))*10)/10) +  ' µm\n')

print('Finger length : ' + str(np.round(np.mean(FingerCharac['Length (µm)'])*10)/10) +
      ' \u00B1 ' + str(np.round(np.std(FingerCharac['Length (µm)']/np.sqrt(len(FingerCharac)))*10)/10) +  ' µm\n')

print('Finger width : ' + str(np.round(np.mean(FingerCharac['Width (µm)'])*10)/10) +
      ' \u00B1 ' + str(np.round(np.std(FingerCharac['Width (µm)']/np.sqrt(len(FingerCharac)))*10)/10) +  ' µm\n')


# Plots

DecoCharac.boxplot(['Angle (deg)','Height (µm)'])
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle('Decorations')



DecoCharac.boxplot('Angle (deg)', 'StackSlice')
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle(None)

DecoCharac.boxplot('Height (µm)', 'StackSlice')
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle(None)

plt.show()

FingerCharac.boxplot(['Length (µm)','Width (µm)'])
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle('Fingers')

FingerCharac.boxplot('Length (µm)', 'StackSlice')
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle(None)

FingerCharac.boxplot('Width (µm)', 'StackSlice')
fig = plt.gcf()
fig.set_dpi(150)
fig.suptitle(None)

FingerCharac.plot(x = 'Width (µm)', y = 'Length (µm)',kind='scatter')
fig = plt.gcf()
fig.set_dpi(150)


# Test Zone